# Churn Data Preprocessing

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## Read the churn file 

In [ ]:
df=pd.read_csv("../data/churn.csv", sep=";")
df


## Data Preparation
### Transform strings into numeric attributes

In [ ]:
X= df.drop(columns=["LEAVE"])
y= df["LEAVE"]

## Label Encoders

In [ ]:
# as an alternative using OrdinalEncoder to make the mapping explicit (STAY=0, LEAVE=1)
leave_encoder = OrdinalEncoder(categories=[["STAY","LEAVE"]],dtype=int).set_output(transform="pandas")
col_encoder = OrdinalEncoder(categories=[["zero","one"]],dtype=int)

## Ordinal Encoders

In [ ]:
# Ordinal encoding for the ordinal attributes (considering change of plan, satisfaction, usage level)
change_encoder = OrdinalEncoder(categories=[['no','never_thought','perhaps', 'considering', 'actively_looking_into_it' ]], dtype=int)
sat_encoder = OrdinalEncoder(categories=[["very_unsat",'unsat','avg','sat','very_sat']], dtype=int)
usage_encoder = OrdinalEncoder(categories=[['very_little','little','avg', 'high', 'very_high']], dtype=int)

## Standardize the final result

In [ ]:
# Scaling the numerical features
col_names_for_scaling = X.columns.values.tolist()
col_names_for_scaling.remove("COLLEGE")
print(col_names_for_scaling)

stsc = StandardScaler().set_output(transform="pandas")


## Put the Encoder and Scaler into a Pipeline

In [ ]:
#combine the preprocessing steps into a pipeline    
column_encoder = ColumnTransformer([("college", col_encoder, ["COLLEGE"]),
                                  ("change", change_encoder, ["CONSIDERING_CHANGE_OF_PLAN"] ),
                                  ("satisfaction", sat_encoder, ["REPORTED_SATISFACTION"] ),
                                  ("usage", usage_encoder, ["REPORTED_USAGE_LEVEL"])],
    remainder='passthrough', verbose_feature_names_out=False).set_output(transform="pandas")
column_scaler = ColumnTransformer ([("scaler", stsc, col_names_for_scaling)],
                                   remainder='passthrough', verbose_feature_names_out=False ).set_output(transform="pandas")
steps = [('column_encoder', column_encoder), ('column_scaler', column_scaler)]
pipe = Pipeline(steps=steps).set_output(transform="pandas")
display(pipe.fit_transform(X))
y = leave_encoder.fit_transform(y.to_frame()[["LEAVE"]])
display(y)
